In [ ]:
import numpy as np
import multiprocessing as mp
import time
import os

Description of the Problem

We consider the problem of estimating the volume of the intersection between two three-dimensional bodies using Monte Carlo integration.

Let the first body be a cube defined by

$$C = [-1,1]^3,$$
and the second body be a sphere centered at the origin with radius $r = 1$, defined by

$$
S = \{(x,y,z) \in \mathbb{R}^3 \mid x^2 + y^2 + z^2 \leq 1\}.
$$

The objective is to estimate the volume

$$
V = \mathrm{Vol}(C \cap S).
$$

Since the sphere is fully contained within the cube, this reduces to estimating the volume of the sphere using uniform sampling over the cube.

Description of the Algorithm

Serial Monte Carlo Algorithm

The Monte Carlo method estimates the volume using random sampling. The procedure is as follows:

- Generate $N$ random points uniformly distributed in the cube $[-1,1]^3$.

- For each point \( (x,y,z) \), evaluate whether it satisfies
    $$
    x^2 + y^2 + z^2 \leq 1.
    $$

- Count the number of points that satisfy this condition, denoted by $N_{\text{in}}$.

- Estimate the volume as
    $$
    V \approx 8 \cdot \frac{N_{\text{in}}}{N},
    $$
    where $8$ is the volume of the cube.


Parallel Algorithm

The algorithm is naturally parallelizable due to the independence of each sampled point. The parallel version proceeds as follows:

- Divide the total number of samples $N$ into $p$ subsets.
- Assign each subset to a different process.
- Each process computes the number of points inside the sphere in its subset.
- Collect the partial results and compute the total number of points inside the sphere:
    $$N_{\text{in}} = \sum_{i=1}^{p} N_{\text{in}}^{(i)}.$$
- Estimate the volume using
    $$V \approx 8 \cdot \frac{N_{\text{in}}}{N}.$$

In [ ]:
def monte_carlo_volume(N):
    points = np.random.uniform(-1, 1, (N, 3)) #Medidas del cubco
    inside = np.sum(np.sum(points**2, axis=1) <= 1)
    return 8 * inside / N #Volume

# Example
N = 1_000_000
vol = monte_carlo_volume(N)
print(f"Estimated Volume: {vol}")

Estimated Volume: 4.188616


In [ ]:
def is_inside_sphere(point):
    return np.sum(np.array(point)**2) <= 1

def test_is_inside_sphere():
    assert is_inside_sphere((0,0,0)) == True
    assert is_inside_sphere((1,0,0)) == True
    assert is_inside_sphere((2,0,0)) == False

test_is_inside_sphere()
print("Unit tests passed.")

Unit tests passed.


Parallelization Strategy and Performance Analysis

Parallelization Strategy

The most computationally intensive part of the algorithm is the evaluation of the condition
$$
x^2 + y^2 + z^2 \leq 1
$$
for a large number of randomly generated points.

Each evaluation is independent of the others, which means that the algorithm is embarrassingly parallel. Therefore, the total workload can be divided into smaller independent tasks and distributed across multiple processes.

Each process performs the following steps:

- Generate a subset of random points.
- Evaluate the condition for each point.
- Count the number of points inside the sphere.


The only required communication occurs after all processes finish, when the partial counts are summed to obtain the total number of points inside the sphere.



In [ ]:
def worker(args):
    n_samples, seed = args

    rng = np.random.default_rng(seed)
    points = rng.uniform(-1, 1, (n_samples, 3))

    inside = np.sum(np.sum(points**2, axis=1) <= 1)
    return inside

def monte_carlo_parallel(N, num_processes):
    # Split N as evenly as possible
    chunk_sizes = [N // num_processes] * num_processes
    for i in range(N % num_processes):
        chunk_sizes[i] += 1

    # Generate different seeds for each process
    seeds = [int(time.time()) + i for i in range(num_processes)]

    tasks = list(zip(chunk_sizes, seeds))

    with mp.Pool(processes=num_processes) as pool:
        results = pool.map(worker, tasks)

    total_inside = sum(results)
    return 8 * total_inside / N

monte_carlo_parallel(1000000, 4)

np.float64(4.19304)

Performance Analysis

Let $T_1$ denote the execution time of the serial algorithm and $T_p$ the execution time using $p$ processes.

The speedup is defined as
$$
S(p) = \frac{T_1}{T_p}.
$$

Ideally, one would expect linear speedup, i.e.,
\[
S(p) \approx p,
\]
but in practice, the speedup is sublinear due to overhead costs such as process creation, memory access, and inter-process communication.

However, for sufficiently large values of $N$, the computational cost dominates the overhead, and parallel execution yields significant improvements in performance.

In [ ]:
if __name__ == "__main__":
    N = 20_000_000  # large for visible speedup
    max_procs = min(8, os.cpu_count())

    print("Serial Execution")
    start = time.time()
    vol_serial = monte_carlo_volume(N)
    t_serial = time.time() - start

    print(f"Volume: {vol_serial}")
    print(f"Time: {t_serial:.4f} s\n")

    print("Parallel Execution")
    for p in range(1, max_procs + 1):
        start = time.time()
        vol_parallel = monte_carlo_parallel(N, p)
        t_parallel = time.time() - start

        speedup = t_serial / t_parallel

        print(f"Processes: {p}")
        print(f"Volume: {vol_parallel}")
        print(f"Time: {t_parallel:.4f} s")
        print(f"Speedup: {speedup:.2f}\n")

Serial Execution
Volume: 4.1896764
Time: 1.7715 s

Parallel Execution
Processes: 1
Volume: 4.1864352
Time: 0.9826 s
Speedup: 1.80

Processes: 2
Volume: 4.1888424
Time: 0.7930 s
Speedup: 2.23

